# Multi class model training

In [ ]:
from sklearn.metrics import cohen_kappa_score
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import random
import matplotlib.pyplot as plt
from tqdm import tqdm
import copy
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score,accuracy_score
from config import device
from transformations import class_weights_tensor,sev_train_loader,sev_val_loader,final_test_ds,get_two_stage_prediction,final_test_loader

In [ ]:
import pickle
binary_model = pickle.load(open("binary_model.pkl", "rb"))

In [ ]:
#Build Severity Model (4 Classes) 
sev_model = models.efficientnet_b0(weights='DEFAULT')

# Replace head: Input features -> 4 outputs (Classes 0,1,2,3 which map to 1,2,3,4)
num_ftrs = sev_model.classifier[1].in_features
sev_model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_ftrs, 4) 
)

sev_model = sev_model.to(device)

In [ ]:
# --- Loss Function (Weighted) ---
# We use the weights calculated in the previous step
criterion_sev = nn.CrossEntropyLoss(weight=class_weights_tensor)

#Optimizer
optimizer_sev = Adam(sev_model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler_sev = CosineAnnealingLR(optimizer_sev, T_max=10)

In [ ]:
#Training Function for Multiclass
def train_severity_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=20, patience=5):
    scaler = torch.cuda.amp.GradScaler()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf')
    best_kappa = -1.0
    counter = 0

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # --- TRAIN ---
        model.train()
        running_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Train Loss: {epoch_loss:.4f}")

        #VAL
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                with torch.cuda.amp.autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                # Get predictions (Index of max logit)
                _, preds = torch.max(outputs, 1)
                
                val_loss += loss.item() * inputs.size(0)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= len(val_loader.dataset)
        
        # Calculate Kappa (Quadratic)
        val_kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
        acc = accuracy_score(all_labels, all_preds)
        
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {acc:.4f} | Val Kappa: {val_kappa:.4f}")

        # Save best model based on Kappa since dataset is imbalanced
        if val_kappa > best_kappa:
            best_kappa = val_kappa
            best_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), 'best_severity_model.pth')
            print("-> Best Severity Model Saved!")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print("Early stopping triggered.")
                break
                
        scheduler_sev.step()

    model.load_state_dict(best_model_wts)
    return model

In [ ]:
print("Severity Model Training...")

sev_model = train_severity_model(
    sev_model,
    sev_train_loader,
    sev_val_loader,
    criterion_sev,
    optimizer_sev,
    num_epochs=30,  # A bit more time for the harder task
    patience=8      # Wait longer before early stopping
)

# Multi class model evaluation

In [ ]:
# Pick a random image from the test set
idx = random.randint(0, len(final_test_ds) - 1)
image_tensor, true_label = final_test_ds[idx]
image_tensor = image_tensor.unsqueeze(0).to(device) # Add batch dimension

#TEST 1: Binary Model Only 
binary_model.eval()
with torch.no_grad():
    bin_logits = binary_model(image_tensor)
    bin_prob = torch.sigmoid(bin_logits).item()
    
print(f"--- Image Index {idx} ---")
print(f"True Label (0-4): {true_label}")
print("-" * 30)
print(f"Model 1 (Binary Only) Output:")
print(f"  Probability of DR: {bin_prob:.4f}")
print(f"  Prediction: {'DR (Disease)' if bin_prob >= 0.5 else 'No_DR (Healthy)'}")

#Combined Pipeline 
final_pred = get_two_stage_prediction(binary_model, sev_model, image_tensor, threshold=0.5)

print("-" * 30)
print(f"Combined Pipeline Output:")
print(f"  Final Prediction (0-4): {final_pred}")

# Show the image
plt.imshow(image_tensor.cpu().squeeze().permute(1, 2, 0) * 0.229 + 0.485) # Denormalize for display
plt.axis('off')
plt.title(f"True: {true_label} | Pred: {final_pred}")
plt.show()

In [ ]:
def evaluate_final_pipeline(binary_model, severity_model, test_loader, device):
    print("Evaluating Final Two-Stage Pipeline on all test images...")
    
    all_preds = []
    all_labels = []
    
    # Ensure eval mode
    binary_model.eval()
    severity_model.eval()
    
    with torch.no_grad():
        for i, (image, label) in enumerate(tqdm(test_loader)):
            image = image.to(device)
            
            # Get prediction using our custom 2-stage function
            pred = get_two_stage_prediction(binary_model, severity_model, image, threshold=0.5)
            
            all_preds.append(pred)
            all_labels.append(label.item())
            
    # Metrics
    acc = accuracy_score(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    
    print("\n" + "="*60)
    print(f"FINAL 2-STAGE PIPELINE RESULTS")
    print("="*60)
    print(f"Final Accuracy: {acc*100:.2f}%")
    print(f"Final Quadratic Kappa: {kappa:.4f}")
    print("-" * 60)
    print("Confusion Matrix (Rows=True, Cols=Pred):")
    print(confusion_matrix(all_labels, all_preds))
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))
    
    return all_labels, all_preds

# Run the evaluation
true_labels, final_preds = evaluate_final_pipeline(binary_model, sev_model, final_test_loader, device)